In [1]:
from pyspark.sql import functions as F

# ============================================================
# PHASE 5.10 - CREATE MANAGED DELTA TABLES
# ============================================================

SourcePath = "Files/AI_Exchange/Output/Phase5_9"

# ------------------------------------------------------------
# 1. Read Parquet files
# ------------------------------------------------------------

FinalMatchingDecisionsDf = spark.read.parquet(
    f"{SourcePath}/FinalMatchingDecisions.parquet"
)

Phi3ValidationResultsDf = spark.read.parquet(
    f"{SourcePath}/Phi3ValidationResults.parquet"
)

FinalMatchingSummaryDf = spark.read.parquet(
    f"{SourcePath}/FinalMatchingSummary.parquet"
)

print("Successfully read all three Parquet files.")

# ------------------------------------------------------------
# 2. Validate source record counts
# ------------------------------------------------------------

ExpectedCounts = {
    "FinalMatchingDecisions": 185,
    "Phi3ValidationResults": 8,
    "FinalMatchingSummary": 10
}

ActualCounts = {
    "FinalMatchingDecisions": FinalMatchingDecisionsDf.count(),
    "Phi3ValidationResults": Phi3ValidationResultsDf.count(),
    "FinalMatchingSummary": FinalMatchingSummaryDf.count()
}

for TableName, ExpectedCount in ExpectedCounts.items():

    ActualCount = ActualCounts[TableName]

    print(f"{TableName}: {ActualCount} rows")

    if ActualCount != ExpectedCount:
        raise ValueError(
            f"Count mismatch for {TableName}. "
            f"Expected {ExpectedCount}, got {ActualCount}"
        )

print("Source record count validation passed.")

# ------------------------------------------------------------
# 3. Create managed Delta tables
# ------------------------------------------------------------

TablesToCreate = {
    "FinalMatchingDecisions": FinalMatchingDecisionsDf,
    "Phi3ValidationResults": Phi3ValidationResultsDf,
    "FinalMatchingSummary": FinalMatchingSummaryDf
}

for TableName, DataFrame in TablesToCreate.items():

    (
        DataFrame.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TableName)
    )

    print(f"Created managed Delta table: {TableName}")

# ------------------------------------------------------------
# 4. Verify Delta tables
# ------------------------------------------------------------

print("\nFINAL DELTA TABLE VALIDATION")
print("=" * 60)

for TableName, ExpectedCount in ExpectedCounts.items():

    TableDf = spark.table(TableName)

    ActualCount = TableDf.count()

    print(f"{TableName}: {ActualCount} rows")

    if ActualCount != ExpectedCount:
        raise ValueError(
            f"Delta table validation failed for {TableName}"
        )

print("=" * 60)
print("PHASE 5.10 COMPLETED SUCCESSFULLY")

StatementMeta(, c3dd068b-3c8a-4547-874b-3df29b5c3857, 3, Finished, Available, Finished, False)

Successfully read all three Parquet files.
FinalMatchingDecisions: 185 rows
Phi3ValidationResults: 8 rows
FinalMatchingSummary: 10 rows
Source record count validation passed.
Created managed Delta table: FinalMatchingDecisions
Created managed Delta table: Phi3ValidationResults
Created managed Delta table: FinalMatchingSummary

FINAL DELTA TABLE VALIDATION
FinalMatchingDecisions: 185 rows
Phi3ValidationResults: 8 rows
FinalMatchingSummary: 10 rows
PHASE 5.10 COMPLETED SUCCESSFULLY


In [2]:
TableNames = [
    "finalmatchingdecisions",
    "phi3validationresults",
    "finalmatchingsummary"
]

for TableName in TableNames:
    TableDf = spark.table(TableName)

    print(f"\nTable: {TableName}")
    print(f"Record Count: {TableDf.count()}")

    TableDf.printSchema()

StatementMeta(, c3dd068b-3c8a-4547-874b-3df29b5c3857, 4, Finished, Available, Finished, False)


Table: finalmatchingdecisions
Record Count: 185
root
 |-- CandidatePairID: string (nullable = true)
 |-- CandidateSchemaVersion: double (nullable = true)
 |-- StyleHaus_ProductID: string (nullable = true)
 |-- UrbanThread_ProductID: string (nullable = true)
 |-- StyleHaus_BatchID: string (nullable = true)
 |-- UrbanThread_BatchID: string (nullable = true)
 |-- StyleHaus_IngestionTimestamp: string (nullable = true)
 |-- UrbanThread_IngestionTimestamp: string (nullable = true)
 |-- StyleHaus_ProductName: string (nullable = true)
 |-- UrbanThread_ProductName: string (nullable = true)
 |-- StyleHaus_Brand: string (nullable = true)
 |-- UrbanThread_Brand: string (nullable = true)
 |-- StyleHaus_Category: string (nullable = true)
 |-- UrbanThread_Category: string (nullable = true)
 |-- StyleHaus_Description: string (nullable = true)
 |-- UrbanThread_Description: string (nullable = true)
 |-- StyleHaus_Price: double (nullable = true)
 |-- UrbanThread_Price: double (nullable = true)
 |-- Styl

In [3]:
from pyspark.sql import functions as F

# ============================================================
# FINAL PHASE 5.10 DATA VALIDATION
# ============================================================

FinalDf = spark.table("finalmatchingdecisions")
Phi3Df = spark.table("phi3validationresults")
SummaryDf = spark.table("finalmatchingsummary")

# 1. Final decision distribution
print("FINAL MATCHING DECISION DISTRIBUTION")

FinalDf.groupBy("FinalMatchDecision_V2") \
    .count() \
    .orderBy("FinalMatchDecision_V2") \
    .show()

# 2. Check duplicate CandidatePairIDs
print("DUPLICATE CANDIDATE PAIR CHECK")

FinalDf.groupBy("CandidatePairID") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# 3. Check null CandidatePairIDs
print("NULL CANDIDATE PAIR ID CHECK")

print(
    "Final table null IDs:",
    FinalDf.filter(F.col("CandidatePairID").isNull()).count()
)

print(
    "Phi3 table null IDs:",
    Phi3Df.filter(F.col("CandidatePairID").isNull()).count()
)

# 4. Phi-3 validation distribution
print("PHI-3 VALIDATION DISTRIBUTION")

Phi3Df.groupBy("Phi3ValidationDecision") \
    .count() \
    .show()

# 5. Summary metrics
print("FINAL MATCHING SUMMARY")

SummaryDf.show(20, truncate=False)

# 6. Verify Delta table format
print("DELTA TABLE FORMAT")

for TableName in [
    "finalmatchingdecisions",
    "phi3validationresults",
    "finalmatchingsummary"
]:
    DetailDf = spark.sql(f"DESCRIBE DETAIL {TableName}")

    DetailDf.select("name", "format", "location").show(
        truncate=False
    )

print("FINAL VALIDATION EXECUTED")

StatementMeta(, c3dd068b-3c8a-4547-874b-3df29b5c3857, 5, Finished, Available, Finished, False)

FINAL MATCHING DECISION DISTRIBUTION
+---------------------+-----+
|FinalMatchDecision_V2|count|
+---------------------+-----+
|                MATCH|   22|
|             NO_MATCH|  155|
|               REVIEW|    8|
+---------------------+-----+

DUPLICATE CANDIDATE PAIR CHECK
+---------------+-----+
|CandidatePairID|count|
+---------------+-----+
+---------------+-----+

NULL CANDIDATE PAIR ID CHECK
Final table null IDs: 0
Phi3 table null IDs: 0
PHI-3 VALIDATION DISTRIBUTION
+----------------------+-----+
|Phi3ValidationDecision|count|
+----------------------+-----+
|                REVIEW|    1|
|                 MATCH|    7|
+----------------------+-----+

FINAL MATCHING SUMMARY
+--------------------------+-----+
|Metric                    |Value|
+--------------------------+-----+
|TotalCandidatePairs       |185  |
|FinalMATCH                |22   |
|FinalNO_MATCH             |155  |
|FinalREVIEW               |8    |
|Phi3ValidatedPairs        |8    |
|Phi3SuggestedMATCH        |